<a href="https://colab.research.google.com/github/MichalSlowakiewicz/Visual-Recognition/blob/master/LAB_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [FBNet](https://arxiv.org/pdf/1812.03443.pdf) on MNIST

In this exercise we will have fun with a simplified version of the [FBNet](https://arxiv.org/pdf/1812.03443.pdf) architecture for neural architecture search (NAS).
We will implement the following parts:
- a *depthwise separable convolution*,
- an FBNet block,
- a NAS experiment, after which we'll visualize a [Pareto front](https://en.wikipedia.org/wiki/Pareto_front) on an accuracy vs. latency scatter plot.

## Imports and dataset

In [1]:
%pip install lightning --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 17.2 MB/s eta 0:00:00


In [2]:
import json
import logging
import time
import warnings
from collections import defaultdict
from pathlib import Path
from typing import cast

import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms.v2 as v2
from lightning.pytorch.loggers import TensorBoardLogger
from torch import Tensor
from torch.utils.data import DataLoader
import statistics

In [3]:
class TipFilter(logging.Filter):
    """Helper to filter out some noise from lightning."""
    def filter(self, record):
        msg = record.getMessage()
        return not ("💡 Tip" in msg or "TPU available" in msg or "CUDA_VISIBLE_DEVICES" in msg)


logging.getLogger("lightning.pytorch.utilities.rank_zero").addFilter(TipFilter())
warnings.filterwarnings("ignore", ".*does not have many workers.*")
warnings.filterwarnings("ignore", r"(?s).*isinstance\(treespec, LeafSpec\)")

In [6]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128

mean, std = (0.1307,), (0.3081,)
transform = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=mean, std=std),
    ]
)

dataset_path = Path("./data/MNIST")
train_dataset = torchvision.datasets.MNIST(
    root=dataset_path,
    train=True,
    transform=transform,
    download=True,
)

val_dataset = torchvision.datasets.MNIST(
    root=dataset_path,
    train=False,
    transform=transform,
    download=True,
)
INPUT_SHAPE = (1, 28, 28)
CLASSES = train_dataset.classes

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 523kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.61MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.16MB/s]


## Depthwise separable convolution

In this task you will implement a [depthwise separable convolution](https://arxiv.org/pdf/1902.00927v2.pdf) module. The idea is to decompose a standard convolution into two consecutive convolutions that decrease the amount of parameters and computations:

- a *depthwise* convolution, which processes each input channel independently (use `Conv2d` with `groups=in_channels`, and  `out_channels=in_channels`, with appropriate `kernel_size` and `padding`),
- a *pointwise* convolution, which processes each feature map pixel independently (with a 1x1 filter size).

In [ ]:
class DepthwiseSeparableConvolution(nn.Module):
    def __init__(
        self, in_channels: int, out_channels: int, kernel_size: int, padding: int
    ) -> None:
        super().__init__()
        # TODO {
        self.depthwise = nn.Conv2d(in_channels=in_channels, groups=in_channels, out_channels=in_channels, kernel_size=kernel_size, padding=padding)
        self.pointwise = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=1)
        # }

    def forward(self, x: Tensor) -> Tensor:
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x


## FBNetBlock

Let's implement the FBNet Block.

In its constructor, it should initiate:
* a module dict with 7 alternative submodules (each with the same `in_channels` and `out_channels`, and with padding that preserves spatial dimensions):
    - Standard convolution with 1x1 kernel,
    - Standard convolution with 3x3 kernel,
    - Standard convolution with 5x5 kernel,
    - Standard convolution with 7x7 kernel,
    - Depthwise separable convolution with 3x3 kernel,
    - Depthwise separable convolution with 5x5 kernel,
    - Depthwise separable convolution with 7x7 kernel,
* a `logits` tensor of size `(7,)` initiated with zeros that are used to compute the preference of each of the submodules above,
* a `latencies` tensor (initialized lazily).

In the ***`profile` method***, given an input tensor `x`, it should:
* run each submodule (as in evaluation) on this tensor, `n_profiling_evals` times,
* register their execution times and compute the latency of each submodule as the median time,
* set `self.latencies` to be a tensor of a shape `(7,)` with the computed latencies.

In the ***`forward` method***, given an input tensor `x`, a temperature parameter `tau`, and a distilation flag `distill`, it should:
* if `distill` is `False` (training):
    1. compute [`gumbel_softmax`](https://pytorch.org/docs/stable/generated/torch.nn.functional.gumbel_softmax.html) weights `m` (see below; Equation 8 from an original [paper](https://arxiv.org/pdf/1812.03443.pdf)) with temperature `tau`,
    2. compute the output of each of submodules on tensor `x` and average these outputs with weights `m` to obtain the `output` tensor,
    3. compute the average latency by averaging over the `latencies` tensor with weights `m` to obtain a `latency` scalar tensor.
    4. return `output, latency`.
* if `distill` is `True` (inference):
    - Select the submodule with the highest `logits` value; return its `latency` and the result of applying this submodule.

**Questions:**
- Why every submodule in this block should have the same `out_channels`?
- Why do we output `latency` as a scalar tensor, instead of just a float?

*Gumbel-softmax* reminder: this is a common trick used to sample a discrete variable (from a categorical distribution) in a differentiable way.
* Given logits $\ell \in \mathbb{R}^d$ and we want to sample $i \in [d]$ according to the distribution $\text{softmax}(\ell)$ and use the vector $\text{onehot}(i) \in \mathbb{R}^d$.
* The resulting $\text{onehot}(i)$ has no gradients wrt $\ell$, so instead we use a reparameterization trick and an approximation:
  1. Sample $g \in \mathbb{R}^d$ i.i.d. from the $\text{Gumbel}(0,1)$ distribution. Then $i = \arg\max(\ell + g)$ can be proved to have the same distribution as $\text{softmax}(\ell)$.
  2. Approximate $\text{onehot}(\arg\max(\ell + g))$ with $\text{softmax}((\ell + g) / \tau)$, which is differentiable. Here $\tau > 0$ is a temperature parameter, as $\tau \to 0$, the approximation becomes exact.
  3. (Alternatively, you could keep the exact $\text{onehot}(\arg\max(\ell + g))$ in the forward pass, but use $\text{softmax}((\ell + g) / \tau)$ in the backward pass to compute gradients. This is called the *straight-through* or *hard* Gumbel-softmax estimator.)
  
[F.gumbel_softmax](https://pytorch.org/docs/stable/generated/torch.nn.functional.gumbel_softmax.html)$(\ell, \tau)$ implements this (with `hard=False` or `True`).

In [ ]:
class FBNetBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        n_profiling_evals: int = 50,
    ) -> None:
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.n_profiling_evals = n_profiling_evals

        Cin, Cout = in_channels, out_channels
        self.fb_modules = nn.ModuleDict(
            {
                # TODO {
                "Conv 1x1": nn.Conv2d(in_channels=self.in_channels, out_channels=self.out_channels, kernel_size=1, padding=0),
                "Conv 3x3": nn.Conv2d(in_channels=self.in_channels, out_channels=self.out_channels, kernel_size=3, padding=1),
                "Conv 5x5": nn.Conv2d(in_channels=self.in_channels, out_channels=self.out_channels, kernel_size=5, padding=2),
                "Conv 7x7": nn.Conv2d(in_channels=self.in_channels, out_channels=self.out_channels, kernel_size=7, padding=3),
                "DSConv 3x3": DepthwiseSeparableConvolution(in_channels=self.in_channels, out_channels=self.out_channels, kernel_size=3, padding=1),
                "DSConv 5x5": DepthwiseSeparableConvolution(in_channels=self.in_channels, out_channels=self.out_channels, kernel_size=5, padding=2),
                "DSConv 7x7": DepthwiseSeparableConvolution(in_channels=self.in_channels, out_channels=self.out_channels, kernel_size=7, padding=3),
                # }
            }
        )

        self.logits = nn.Parameter(torch.zeros(size=(len(self.fb_modules),)))

        # Latency of each fb_module (in seconds). These are not learnable params,
        # but we want to store them in state_dict, hence we register it as a buffer.
        self.latencies: Tensor
        self.register_buffer("latencies", torch.zeros(size=(len(self.fb_modules),)))
        self.profiled = False

    def forward(
        self, x: Tensor, tau: float = 1.0, distill: bool = False
    ) -> tuple[Tensor, Tensor]:
        """
        Input: image or feature map of shape (B, in_channels, H, W).
        Output: a tuple of (output, latency) where:
        * `output` is the processed tensor of shape (B, out_channels, H, W) (weighted average of submodule outputs),
        * `latency` is the latency scalar tensor (weighted average of submodule latencies).
        """
        if not self.profiled:
            self.profile(x)

        # TODO {
        if distill:
          id_max = torch.argmax(self.logits)
          layer_max = list(self.fb_modules.values())[id_max]
          output = layer_max(x)
          lat = self.latencies[id_max]
          return (output, lat)
        else:
          gum = F.gumbel_softmax(logits=self.logits, tau=tau, dim=-1)
          sum_output = 0
          sum_lats = 0
          outputs = [weight*layer(x) for weight, layer in zip(gum, list(self.fb_modules.values()))]
          sum_output = sum(outputs)

          lats = [weight*lat for weight, lat in zip(gum, self.latencies)]
          sum_lats = sum(lats)
          return (sum_output, sum_lats)

        # }

    def profile(self, x: Tensor, device: torch.device | None = None) -> Tensor:
        # TODO {
        medians = []
        for layer in self.fb_modules.values():
          layer_times = []

          for _ in range(self.n_profiling_evals):
            start = time.perf_counter()
            output = layer(x)
            end = time.perf_counter()
            tim = end - start
            layer_times.append(tim)
          median = statistics.median(layer_times)
          medians.append(median)
        self.latencies = torch.tensor(medians, device=device or x.device, dtype=torch.float32)
        # }
        self.profiled = True
        return self.latencies

    def get_submodule_probs(self) -> dict[str, float]:
        """Get map from submodule name to its probability."""
        probs = F.softmax(self.logits, dim=0).cpu()
        return {
            module_name: prob.item()
            for module_name, prob in zip(self.fb_modules.keys(), probs)
        }

    def get_submodule_latencies(self) -> dict[str, float]:
        """Get map from submodule name to its latency (in seconds)."""
        return {
            module_name: latency.item()
            for module_name, latency in zip(self.fb_modules.keys(), self.latencies)
        }

## FBNetModel

The FBNetModel consists of a sequence of FBNetBlocks. It's `forward` method should apply them consecutively to an input tensor `x` and return the final `output` together with the total `latency` from all blocks.
Use:
* a `relu` activation after each block,
* a pooling operation (downsizing spatial dimensions by two) after the first two blocks,
* a dense layer to obtain `out_channels` logits.

In [ ]:
class FBNetModel(nn.Module):
    def __init__(self, input_shape: tuple[int, int, int], num_classes: int) -> None:
        super().__init__()
        self.input_shape = input_shape
        self.num_classes = num_classes
        C, H, W = input_shape
        self.blocks = nn.ModuleList(
            [
                FBNetBlock(in_channels=C, out_channels=16),
                FBNetBlock(in_channels=16, out_channels=32),
                FBNetBlock(in_channels=32, out_channels=64),
            ]
        )
        self.pool = torch.nn.MaxPool2d(2, stride=2)

        self.dense = nn.Linear(64 * (H // 4) * (W // 4), num_classes)

    def forward(
        self, x: Tensor, tau: float = 1.0, distill: bool = False
    ) -> tuple[Tensor, Tensor]:
        """
        Input: image or feature map of shape (B, in_channels, H, W).
        Output: a tuple of (output, latency) where:
        * `output` is the processed tensor of shape (B, out_channels) (logits),
        * `latency` is the latency scalar tensor (sum of latencies from each block).
        """
        # TODO {
        total_latency = 0

        (x, lat) = self.blocks[0](x, tau=tau, distill=distill)
        total_latency += lat
        x = F.relu(x)
        x = self.pool(x)
        (x, lat) = self.blocks[1](x, tau=tau, distill=distill)
        total_latency += lat
        x = F.relu(x)
        x = self.pool(x)
        (x, lat) = self.blocks[2](x, tau=tau, distill=distill)
        total_latency += lat
        x = F.relu(x)
        x = torch.flatten(x, start_dim=1)
        x = self.dense(x)
        return (x, total_latency)
        # }


    def profile(self, device: torch.device | None = None) -> None:
        C, H, W = self.input_shape
        for i, block in enumerate(cast(list[FBNetBlock], self.blocks)):
            block.profile(
                torch.randn(BATCH_SIZE, block.in_channels, H // 2**i, W // 2**i),
                device=device,
            )

    def get_fb_block_probs(self) -> list[dict[str, float]]:
        """Get list of maps from submodule name to its probability, within each FBNetBlock."""
        return [cast(FBNetBlock, block).get_submodule_probs() for block in self.blocks]

    def get_fb_block_latencies(self) -> list[dict[str, float]]:
        """Get list of maps from submodule name to its latency, within each FBNetBlock."""
        return [
            cast(FBNetBlock, block).get_submodule_latencies() for block in self.blocks
        ]

    def plot_fb_block_probs(self, title: str = "") -> None:
        """Plot the probabilities of selecting each submodule within each FBNetBlock."""
        all_probs = self.get_fb_block_probs()
        block_names = [f"FBNetBlock {i + 1}" for i in range(len(all_probs))]
        module_names = list(all_probs[0].keys())

        probs = np.array([[p[n] for n in module_names] for p in all_probs])
        plt.imshow(probs)

        for y in range(probs.shape[0]):
            for x in range(probs.shape[1]):
                plt.text(x - 0.25, y, f"{probs[y, x]:.2%}")

        plt.yticks(list(range(len(block_names))), block_names)
        plt.xticks(
            list(range(len(module_names))), [s.replace(" ", "\n") for s in module_names]
        )
        plt.title(title)
        plt.show()

In [ ]:
def example_latency():
    model = FBNetModel(INPUT_SHAPE, num_classes=len(CLASSES))
    model.profile(device=torch.device("cpu"))

    latencies = model.get_fb_block_latencies()
    df = pd.DataFrame({f"FBNetBlock {i + 1}": lat for i, lat in enumerate(latencies)})
    display(df.mul(1000).style.set_caption("Latency [ms]").format(precision=2))


example_latency()

## Lightning module
Implement
- a `training_step` where you:
     - compute the loss of your model that is a sum of [`CrossEntropyLoss`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) and `log(latency)` multiplied by a weight `latency_weight`,
     - apply a temperature decay where after each training step you decrease the temperature `tau` by a factor of `1 - 1e-3`. An initial value of a `tau` parameter should be set to `1.0`.
     - log `accuracy` and `latency` after each step.
- a `validation_step` where you: validate your distilled (`distill` flag set to `True`) model by computing and logging its `accuracy` and `latency`.


You can use the code below to debug your model. It trains model for 10 epochs, where for the first 5 epochs we train architecture (by training the logits of the `FBNetModule` modules) and later we only train the distilled, best model. We additionally plot architecture logits used for distilation so you can see which operations were selected.

In [ ]:
class LightningModule(L.LightningModule):
    def __init__(self, latency_weight: float = 1.0, distill_after: int = 5) -> None:
        super().__init__()
        self.model = FBNetModel(input_shape=INPUT_SHAPE, num_classes=len(CLASSES))
        self.temperature = 1.0
        self.latency_weight = latency_weight
        self.distill = False
        self.distill_after = distill_after

    def on_train_start(self) -> None:
        self.temperature = 1.0
        # Profile on CPU (you can train on GPU, but still optimize for time of inference on CPU).
        C, H, W = INPUT_SHAPE
        for i, block in enumerate(cast(list[FBNetBlock], self.model.blocks)):
            block.profile(
                torch.randn(BATCH_SIZE, block.in_channels, H // 2**i, W // 2**i),
                device=torch.device("cpu"),
            )

    def on_train_epoch_end(self) -> None:
        if self.current_epoch == (self.distill_after - 1):
            print("Starting distillation")
            self.distill = True
            self.model.plot_fb_block_probs(f"latency_weight={self.latency_weight}")

    def training_step(self, batch: tuple[Tensor, Tensor], batch_idx: int) -> Tensor:
        # TODO {
        x, y = batch

        # 1. Przejście w przód (z uwzględnieniem trybu distill i temperatury)
        logits, latency = self.model(x, tau=self.temperature, distill=self.distill)

        # 2. Obliczenie błędu klasyfikacji
        ce_loss = F.cross_entropy(logits, y)

        # 3. Całkowity błąd (NAS Loss = CrossEntropy + Waga * Opóźnienie)
        loss = ce_loss + self.latency_weight * latency

        # Logowanie (opcjonalne, ale bardzo użyteczne w PyTorch Lightning)
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_ce_loss", ce_loss)
        self.log("train_latency", latency)

        return loss

    def validation_step(self, batch: tuple[Tensor, Tensor], batch_idx: int) -> None:
        # TODO {
        x, y = batch

        # 1. Przejście w przód
        logits, latency = self.model(x, tau=self.temperature, distill=self.distill)

        # 2. Obliczenia błędów i dokładności (Accuracy) dla statystyk
        val_loss = F.cross_entropy(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()

        # Logowanie wyników walidacji (np. do paska postępu)
        self.log("val_loss", val_loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        self.log("val_latency", latency)
        # }

    def configure_optimizers(self) -> torch.optim.Optimizer:
        return torch.optim.AdamW(self.parameters(), lr=1e-3)


## Training
One epoch should take ~40s on a Colab T4 GPU.

In [ ]:
pl_module = LightningModule(latency_weight=0.5)

In [ ]:
trainer = L.Trainer(
    max_epochs=10,
    logger=TensorBoardLogger("runs/fb_net", name=f"lat_{pl_module.latency_weight}"),
)

In [ ]:
trainer.fit(pl_module, train_loader, val_loader)

In [ ]:
trainer.validate(pl_module, val_loader);

In [ ]:
L.Trainer(accelerator="cpu").validate(pl_module.to("cpu"), val_loader);

## Accuracy vs latency and the Pareto front

The following code would train & evaluate models for each of preselected `LATENCY_WEIGHTS` and plot their `accuracy` and `latency`.
To avoid waiting for all that, let's just download pre-computed results (these are for a non-Colab CPU).

Make a scatter plot of `accuracy` vs `latency` with different colors for different `latency_weight` values.

In [ ]:
# LATENCY_WEIGHTS = [0.0, 0.02, 0.05, 0.1, 0.5, 1.0, 10.0]

# # Map from latency weight to list of validation results (each result is a dict mapping metric name to value).
# all_results = defaultdict[float, list[dict[str, float]]](list)

In [ ]:
# for repeat in range(10):
#     for latency_weight in LATENCY_WEIGHTS:
#         print("#" * 50, f"{repeat=}, {latency_weight=}", "#" * 50)
#         trainer = L.Trainer(
#             max_epochs=10,
#             enable_model_summary=False,
#             logger=TensorBoardLogger("runs/fb_net", name=f"lat_{latency_weight}"),
#         )
#         pl_module = LightningModule(latency_weight=latency_weight)
#         trainer.fit(pl_module, train_loader, val_loader)
#         results = trainer.validate(pl_module, dataloaders=val_loader, verbose=False)[0]
#         all_results[latency_weight].append(dict(results))
#         with open("all_results.json", "w") as f:
#             json.dump(dict(all_results), f)

In [ ]:
!wget --quiet https://mimuw.edu.pl/~mwrochna/upload/H_FBNet_results.json

In [ ]:
# Map from latency weight to list of validation results (each result is a dict mapping metric name to value).
all_results: dict[str, list[dict[str, float]]]

with open("H_FBNet_results.json") as f:
    all_results = json.load(f)

In [ ]:
print(all_results.keys())

In [ ]:
print(all_results["0.0"][0].keys())

In [ ]:
def show_scatter_plot(all_results: dict[str, list[dict[str, float]]], x: str = "val_acc", y: str = "val_latency") -> None:
    # TODO {
    # }


show_scatter_plot(all_results, "actual_latency", "val_latency")
show_scatter_plot(all_results, "val_acc", "val_latency")

Questions:
- Which regions of the presented plot might be considered as the best?
- Does the `latency_weight` influence the final latency of your model?
- Does it affect the `accuracy`?
- Which operations are selected for each of the `latency_weight`, does the `DepthwiseSeparableConvolution` looks like optimal choice for a fast computations in `pytorch`?
- What's your intuition why repeated optimization might converge to different (and sometimes couterintuitive) solutions? You may witness the infamous [Matthew effect](https://en.wikipedia.org/wiki/Matthew_effect#:~:text=The%20Matthew%20effect%20of%20accumulated,and%20the%20poor%20get%20poorer%22.).

## Tensorboard

In [ ]:
%load_ext tensorboard
!mkdir -p runs/fb_net
%tensorboard --logdir runs/fb_net